# 06 — Error analysis & diagnostic reporting

**Phase:** After selecting a **champion regime** (here: **logistic regression** on the **credits-enriched** feature matrix from notebook 05), aggregate test metrics are **necessary but not sufficient**. This notebook is the **diagnostic / reporting** layer: where the model is **right**, where it **fails**, and whether **talent/credits** signals change the *shape* of errors.

**Workflow:** objective → rebuild aligned predictions on the **same holdout** as notebook 05 → transition analysis (baseline vs credits) → class- and segment-level views → confidence diagnostics → **case cards** for product / Streamlit storytelling → honest business synthesis.

**Style:** **Observation → Business interpretation → Modeling implication** (consistent with notebooks 01–05).

**Setting:** **Pre-release** features only; this is **triage intelligence**, not an oracle.


## 1. Objective — why we go beyond accuracy

Portfolio models are used to **structure conversations** (“what did the model see?”), not to replace judgment. **Aggregate F1** answers “how noisy is the overall scorecard?” — it does **not** answer:

- Which **commercial segments** (genre, scale, franchise packaging) are systematically misunderstood?
- Did credits features **fix** specific failure modes or simply **re-shuffle** errors?
- When the model is **confident but wrong**, what operational risk does that create?

This notebook therefore **joins predictions to business covariates** on the held-out test movies and quantifies **transitions**, **segment performance**, and **confidence behavior**.

---

**Observation:** A model can lift macro-F1 while leaving some segments untouched — or worse.  
**Business interpretation:** Executives need **segmented truth**, not a single leaderboard number.  
**Modeling implication:** Error analysis should precede **UI design** (e.g. Streamlit) and **calibration** investments.


## 2. Load data, reproduce champion split, build the analysis dataset

We reload the **same sources** as notebook 05 and rebuild the **identical** stratified split (`random_state=42`, `test_size=0.2`). We fit **two** multinomial **logistic regression** pipelines:

1. **Baseline** — notebook 03 feature contract.  
2. **Credits** — credits-enriched matrix from notebook 05.

We then assemble **`analysis_df`** on **test rows only** with labels, both predictions, **credits** `predict_proba` (per-class columns), and interpretable metadata (genre, budget, runtime, production scale, talent score, cast/crew, franchise flag, etc.).

---

**Observation:** Diagnostics must be **row-aligned** with transparent provenance.  
**Business interpretation:** A diagnostic table is the bridge between **model output** and **creative/finance language**.  
**Modeling implication:** Keep the champion split fixed so this notebook is **comparable** to 05.


In [ ]:
from __future__ import annotations

import ast
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

_CWD = Path.cwd().resolve()
if (_CWD / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD
elif (_CWD.parent / "data" / "processed").is_dir():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD
    print("Warning: data/processed not found; using cwd as PROJECT_ROOT.")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "movies_cleaned_with_target.csv"
CREDITS_PATH = PROJECT_ROOT / "data" / "raw" / "tmdb_5000_credits.csv"
PLOTS_DIR = PROJECT_ROOT / "plots" / "modeling"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk", font_scale=0.95)
plt.rcParams["figure.figsize"] = (11, 5.5)
plt.rcParams["axes.titlesize"] = 14


def save_fig(name: str) -> Path:
    path = PLOTS_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white", edgecolor="none")
    plt.close()
    return path


TARGET_COLUMN = "movie_success_class"
FORBIDDEN_IN_X = {"revenue", "roi", "log_roi", TARGET_COLUMN}

BASELINE_CANDIDATES = [
    "budget",
    "runtime",
    "main_genre",
    "original_language",
    "release_month",
    "release_quarter",
    "genre_count",
    "production_company_count",
    "production_country_count",
    "spoken_language_count",
]
STATIC_ENGINEERED = [
    "budget_log",
    "runtime_bucket",
    "international_production",
    "multilingual_movie",
    "release_season",
    "genre_complexity",
    "decade",
]
ENGINEERED_FOR_MODEL = STATIC_ENGINEERED + ["production_scale"]
CREDITS_NUMERIC = [
    "director_movie_count",
    "top_director_flag",
    "top_billed_cast_count",
    "known_actor_count",
    "cast_size",
    "crew_size",
    "writer_count",
    "possible_franchise_flag",
    "ensemble_cast_flag",
    "talent_score",
]
CREDITS_CATEGORICAL = ["director_bucket"]
TOP_DIRECTORS_K = 40
TOP_ACTORS_K = 120
DIRECTOR_BUCKET_TOP_N = 25

analysis_df = None
classes_order = None
pipe_baseline = pipe_credits = None

df = df_credits = df_join = None
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
if CREDITS_PATH.exists():
    df_credits = pd.read_csv(CREDITS_PATH)

if df is not None and df_credits is not None:
    try:
        df_join = df.merge(
            df_credits,
            left_on="id",
            right_on="movie_id",
            how="left",
            suffixes=("", "_cred"),
            validate="one_to_one",
        )
    except Exception:
        df_join = df.merge(
            df_credits,
            left_on="id",
            right_on="movie_id",
            how="left",
            suffixes=("", "_cred"),
        )
    if "movie_id" in df_join.columns:
        df_join = df_join.drop(columns=["movie_id"])
    print("Joined shape:", df_join.shape)
else:
    print("Missing movies and/or credits — cannot build analysis_df.")


In [ ]:
def safe_json_list(val) -> list:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        s = val.strip()
        if not s:
            return []
        try:
            return json.loads(s)
        except Exception:
            try:
                return ast.literal_eval(s)
            except Exception:
                return []
    return []


def extract_director_name(crew_raw) -> str:
    for job in ("Director", "Co-Director"):
        for c in safe_json_list(crew_raw):
            if not isinstance(c, dict):
                continue
            if str(c.get("job", "")) == job:
                n = c.get("name")
                if n:
                    return str(n).strip()
    return "__missing__"


def _order_key(x: dict) -> int:
    o = x.get("order", 999)
    try:
        return int(o)
    except (TypeError, ValueError):
        return 999


def extract_cast_ordered(cast_raw) -> list[dict]:
    lst = [x for x in safe_json_list(cast_raw) if isinstance(x, dict)]
    lst.sort(key=_order_key)
    return lst


def writer_count_from_crew(crew_raw) -> int:
    n = 0
    for c in safe_json_list(crew_raw):
        if not isinstance(c, dict):
            continue
        dep = str(c.get("department", ""))
        job = str(c.get("job", "")).lower()
        if dep == "Writing" or "writer" in job:
            n += 1
    return n


_FRANCHISE_PAT = re.compile(
    r"\b(part|chapter|returns|begins|rise|sequel|prequel|reloaded|revolutions|legacy)\b|"
    r"\b(ii|iii|iv|v|vi|vii|viii|ix)\b|"
    r"\bvs\.?\b|"
    r"\s2$|\s3$|:?\s2\b|:?\s3\b|\b2:\s|\b3:\s",
    re.I,
)


def franchise_heuristic(title, keywords_raw) -> int:
    parts = []
    if title is not None and not (isinstance(title, float) and pd.isna(title)):
        parts.append(str(title))
    for kw in safe_json_list(keywords_raw):
        if isinstance(kw, dict) and kw.get("name"):
            parts.append(str(kw["name"]))
    blob = " ".join(parts).lower()
    return int(bool(_FRANCHISE_PAT.search(blob)))


def production_scale_from_train(train_counts: pd.Series, apply_counts: pd.Series) -> pd.Series:
    t = pd.to_numeric(train_counts, errors="coerce").dropna()
    if t.empty:
        q1, q2 = 0.0, 1.0
    else:
        q1, q2 = t.quantile(1 / 3), t.quantile(2 / 3)
    if q1 == q2:
        q2 = q2 + 1e-6

    def bucket(x):
        if pd.isna(x):
            return "__missing__"
        xv = float(x)
        if xv <= q1:
            return "indie"
        if xv <= q2:
            return "mid_scale"
        return "large_scale"

    return apply_counts.map(bucket)


def apply_train_only_talent_features(df_all: pd.DataFrame, idx_train: np.ndarray, idx_test: np.ndarray) -> pd.DataFrame:
    out = df_all.copy()
    tr = out.iloc[idx_train]
    te = out.iloc[idx_test]
    tr_co = tr["production_company_count"]
    out.loc[tr.index, "production_scale"] = production_scale_from_train(tr_co, tr["production_company_count"]).astype(str)
    out.loc[te.index, "production_scale"] = production_scale_from_train(tr_co, te["production_company_count"]).astype(str)

    dir_counts = tr.loc[tr["director_name"] != "__missing__", "director_name"].value_counts()
    top_directors = set(dir_counts.nlargest(TOP_DIRECTORS_K).index)
    actor_counts: dict[str, int] = {}
    for _, row in tr.iterrows():
        cl = extract_cast_ordered(row.get("cast"))
        seen = set()
        for ent in cl[:12]:
            nm = ent.get("name")
            if not nm:
                continue
            nm = str(nm).strip()
            if nm in seen:
                continue
            seen.add(nm)
            actor_counts[nm] = actor_counts.get(nm, 0) + 1
    top_actors = set(sorted(actor_counts, key=lambda k: actor_counts[k], reverse=True)[:TOP_ACTORS_K])
    dir_movie_count_map = dir_counts.to_dict()
    top_dir_names = set(dir_counts.nlargest(DIRECTOR_BUCKET_TOP_N).index)

    def map_dir_count(name):
        return 0 if name == "__missing__" else int(dir_movie_count_map.get(name, 0))

    def map_top_flag(name):
        return int(name in top_directors and name != "__missing__")

    def map_bucket(name):
        if name == "__missing__":
            return "__missing__"
        if name in top_dir_names:
            return name
        return "__other__"

    def map_known_actor(cast_raw) -> int:
        k = 0
        cl = extract_cast_ordered(cast_raw)
        seen = set()
        for ent in cl[:12]:
            nm = ent.get("name")
            if not nm:
                continue
            nm = str(nm).strip()
            if nm in seen:
                continue
            seen.add(nm)
            if nm in top_actors:
                k += 1
        return k

    out["director_movie_count"] = out["director_name"].map(map_dir_count)
    out["top_director_flag"] = out["director_name"].map(map_top_flag)
    out["director_bucket"] = out["director_name"].map(map_bucket)
    out["known_actor_count"] = out["cast"].map(map_known_actor)
    mm = MinMaxScaler()
    tr_x = tr.assign(_ka=out.loc[tr.index, "known_actor_count"], _dm=out.loc[tr.index, "director_movie_count"])[["_ka", "_dm"]].astype(float)
    mm.fit(tr_x)
    both = out.assign(_ka=out["known_actor_count"], _dm=out["director_movie_count"])[["_ka", "_dm"]].astype(float)
    scaled = mm.transform(both)
    top_flag = out["top_director_flag"].astype(float).values
    out["talent_score"] = 0.45 * scaled[:, 0] + 0.35 * scaled[:, 1] + 0.20 * top_flag
    return out


def prepare_X(feature_cols: list[str], frame: pd.DataFrame):
    X = frame[feature_cols].copy()
    baseline_cat = ["main_genre", "original_language", "release_month", "release_quarter"]
    extra_cat = ["runtime_bucket", "production_scale", "release_season", "genre_complexity", "decade"]
    credits_cat = list(CREDITS_CATEGORICAL)
    cat_cols = [c for c in baseline_cat + extra_cat + credits_cat if c in X.columns]
    num_cols = [c for c in feature_cols if c not in cat_cols]
    for col in ["main_genre", "original_language"]:
        if col in X.columns:
            X[col] = X[col].astype("string").fillna("__missing__")
    for col in ["release_month", "release_quarter"]:
        if col in X.columns:
            X[col] = X[col].apply(lambda v: "__missing__" if pd.isna(v) else str(int(v)))
    for col in cat_cols:
        X[col] = X[col].astype(str)
    return X, num_cols, cat_cols


def build_preprocessor(num_cols: list[str], cat_cols: list[str]) -> ColumnTransformer:
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    tr = []
    if num_cols:
        tr.append(("num", num_pipe, num_cols))
    if cat_cols:
        tr.append(("cat", cat_pipe, cat_cols))
    return ColumnTransformer(transformers=tr)


def make_lr_pipe(num_cols, cat_cols):
    return Pipeline(
        [
            ("prep", build_preprocessor(num_cols, cat_cols)),
            (
                "clf",
                LogisticRegression(
                    max_iter=3000,
                    class_weight="balanced",
                    random_state=42,
                    solver="lbfgs",
                ),
            ),
        ]
    )


if df_join is None:
    print("Stop: no joined dataframe.")
else:
    d = df_join.copy()
    directors, cast_sizes, crew_sizes, writers, top_billed, franch, ensemble = [], [], [], [], [], [], []
    for _, row in d.iterrows():
        cr, ca = row.get("crew"), row.get("cast")
        directors.append(extract_director_name(cr))
        cl = extract_cast_ordered(ca)
        cast_sizes.append(len(cl))
        crew_sizes.append(len(safe_json_list(cr)))
        writers.append(writer_count_from_crew(cr))
        billed = sum(1 for ent in cl[:5] if ent.get("name"))
        top_billed.append(billed)
        franch.append(franchise_heuristic(row.get("title"), row.get("keywords")))
        ensemble.append(int(len(cl) >= 12))
    d["director_name"] = directors
    d["cast_size"] = cast_sizes
    d["crew_size"] = crew_sizes
    d["writer_count"] = writers
    d["top_billed_cast_count"] = top_billed
    d["possible_franchise_flag"] = franch
    d["ensemble_cast_flag"] = ensemble

    d["budget_log"] = np.log1p(d["budget"].clip(lower=0))

    def _runtime_bucket(v) -> str:
        if pd.isna(v):
            return "__missing__"
        r = float(v)
        if r < 90:
            return "short"
        if r <= 120:
            return "medium"
        return "long"

    d["runtime_bucket"] = d["runtime"].map(_runtime_bucket)
    d["international_production"] = (d["production_country_count"].fillna(0) > 1).astype(int)
    d["multilingual_movie"] = (d["spoken_language_count"].fillna(0) > 1).astype(int)

    def _season(m) -> str:
        if pd.isna(m):
            return "__missing__"
        mi = int(m)
        if mi in (12, 1, 2):
            return "winter"
        if mi in (3, 4, 5):
            return "spring"
        if mi in (6, 7, 8):
            return "summer"
        if mi in (9, 10, 11):
            return "fall"
        return "__missing__"

    d["release_season"] = d["release_month"].map(_season) if "release_month" in d.columns else "__missing__"

    def _gc(gc) -> str:
        if pd.isna(gc):
            return "__missing__"
        g = int(gc)
        if g <= 1:
            return "focused"
        if g <= 3:
            return "mixed"
        return "hybrid"

    d["genre_complexity"] = d["genre_count"].map(_gc)
    if "release_year" in d.columns:
        yr = pd.to_numeric(d["release_year"], errors="coerce")

        def _dec(y):
            if pd.isna(y):
                return "__missing__"
            yi = int(y)
            return f"{(yi // 10) * 10}s"

        d["decade"] = yr.map(_dec)
    else:
        d["decade"] = "__missing__"
    d["production_scale"] = "__pending__"

    y_all = d[TARGET_COLUMN].astype(str)
    idx = np.arange(len(d))
    idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=y_all)
    df_model = apply_train_only_talent_features(d, idx_train, idx_test)

    present_base = [c for c in BASELINE_CANDIDATES if c in df_model.columns]
    BASELINE_FEATURES = [c for c in present_base if c not in FORBIDDEN_IN_X]
    ENRICHED_FEATURES = BASELINE_FEATURES + [c for c in ENGINEERED_FOR_MODEL if c in df_model.columns]
    CREDITS_FEATURES = ENRICHED_FEATURES + [c for c in CREDITS_NUMERIC + CREDITS_CATEGORICAL if c in df_model.columns]

    y_train = y_all.iloc[idx_train]
    y_test = y_all.iloc[idx_test]

    def slice_X(cols):
        Xf, nc, cc = prepare_X(cols, df_model)
        return Xf.iloc[idx_train], Xf.iloc[idx_test], nc, cc

    Xtr_b, Xte_b, nb, cb = slice_X(BASELINE_FEATURES)
    Xtr_c, Xte_c, ncr, ccr = slice_X(CREDITS_FEATURES)

    pipe_baseline = make_lr_pipe(nb, cb)
    pipe_credits = make_lr_pipe(ncr, ccr)
    pipe_baseline.fit(Xtr_b, y_train)
    pipe_credits.fit(Xtr_c, y_train)

    pred_b = pipe_baseline.predict(Xte_b)
    pred_c = pipe_credits.predict(Xte_c)
    proba_b = pipe_baseline.predict_proba(Xte_b)
    proba_c = pipe_credits.predict_proba(Xte_c)
    classes_b = list(pipe_baseline.named_steps["clf"].classes_)
    classes_order = list(pipe_credits.named_steps["clf"].classes_)
    assert classes_b == classes_order, "Class order mismatch between baseline and credits pipelines."

    te_meta = df_model.iloc[idx_test].copy()
    title_col = "title" if "title" in te_meta.columns else "original_title"
    titles = te_meta[title_col].astype(str) if title_col in te_meta.columns else te_meta["id"].astype(str)

    analysis_df = pd.DataFrame(
        {
            "movie_id": te_meta["id"].values,
            "title": titles.values,
            "y_true": y_test.values,
            "y_pred_baseline": pred_b,
            "y_pred_credits": pred_c,
        }
    )
    for j, cname in enumerate(classes_order):
        analysis_df[f"p_{cname}"] = proba_c[:, j]
        analysis_df[f"base_p_{cname}"] = proba_b[:, j]

    analysis_df["confidence"] = proba_c.max(axis=1)
    # probability assigned to the *true* class (both models)
    idx_map = {c: i for i, c in enumerate(classes_order)}
    analysis_df["p_true_credits"] = [proba_c[i, idx_map[t]] for i, t in enumerate(analysis_df["y_true"])]
    analysis_df["p_true_baseline"] = [proba_b[i, idx_map[t]] for i, t in enumerate(analysis_df["y_true"])]
    analysis_df["delta_p_true"] = analysis_df["p_true_credits"] - analysis_df["p_true_baseline"]
    analysis_df["predicted_class"] = pred_c
    mg = te_meta["main_genre"].astype(str) if "main_genre" in te_meta.columns else "__missing__"
    analysis_df["main_genre"] = mg.values
    analysis_df["budget"] = te_meta["budget"].values
    analysis_df["runtime"] = te_meta["runtime"].values
    analysis_df["production_scale"] = te_meta["production_scale"].astype(str).values
    analysis_df["talent_score"] = te_meta["talent_score"].values
    analysis_df["cast_size"] = te_meta["cast_size"].values
    analysis_df["crew_size"] = te_meta["crew_size"].values
    analysis_df["possible_franchise_flag"] = te_meta["possible_franchise_flag"].values
    analysis_df["director_name"] = te_meta["director_name"].astype(str).values
    analysis_df["known_actor_count"] = te_meta["known_actor_count"].values

    bl_train = np.log1p(df_model.iloc[idx_train]["budget"].clip(lower=0))
    q1, q2 = bl_train.quantile(1 / 3), bl_train.quantile(2 / 3)

    def budget_bucket(v):
        if pd.isna(v):
            return "__missing__"
        x = float(np.log1p(max(v, 0)))
        if x <= q1:
            return "budget_low"
        if x <= q2:
            return "budget_mid"
        return "budget_high"

    analysis_df["budget_bucket"] = [budget_bucket(v) for v in te_meta["budget"].values]
    analysis_df["runtime_bucket"] = te_meta["runtime_bucket"].astype(str).values

    analysis_df["baseline_correct"] = (analysis_df["y_true"] == analysis_df["y_pred_baseline"]).astype(int)
    analysis_df["credits_correct"] = (analysis_df["y_true"] == analysis_df["y_pred_credits"]).astype(int)

    def transition_row(r):
        b, c = bool(r["baseline_correct"]), bool(r["credits_correct"])
        if b and c:
            return "stable_correct"
        if (not b) and (not c):
            return "stable_wrong"
        if (not b) and c:
            return "wrong_to_correct"
        return "correct_to_wrong"

    analysis_df["transition"] = analysis_df.apply(transition_row, axis=1)

    print("analysis_df shape:", analysis_df.shape)
    print("Test accuracy baseline LR:", round(accuracy_score(analysis_df["y_true"], analysis_df["y_pred_baseline"]), 3))
    print("Test accuracy credits LR:", round(accuracy_score(analysis_df["y_true"], analysis_df["y_pred_credits"]), 3))
    display(analysis_df.head(3))


## 3. Transition analysis — when credits change the answer

We classify each test movie into four **transition buckets** comparing **baseline vs credits** logistic predictions:

| Bucket | Meaning |
|--------|---------|
| **stable_correct** | Both models right |
| **stable_wrong** | Both models wrong |
| **wrong_to_correct** | Baseline wrong → credits right (**win**) |
| **correct_to_wrong** | Baseline right → credits wrong (**regression**) |

We quantify volumes, slice by **true label**, and contrast **talent_score**, **cast_size**, **franchise flag**, and **budget** across buckets.

---

**Observation:** Net metric lift can hide **trade-offs** (helping one segment while hurting another).  
**Business interpretation:** “Improved ranking” should be narrated with **who paid the price**.  
**Modeling implication:** If regressions concentrate in sparse segments, consider **constraints** or **segment-specific** thresholds next.


In [ ]:
if analysis_df is None:
    print("Skip transition analysis.")
else:
    vc = analysis_df["transition"].value_counts()
    print("\n===== TRANSITION COUNTS (TEST) =====")
    display(vc.to_frame("n_movies"))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    order = ["stable_correct", "wrong_to_correct", "stable_wrong", "correct_to_wrong"]
    order = [o for o in order if o in vc.index]
    sns.barplot(x=vc.reindex(order).index, y=vc.reindex(order).values, ax=ax, color="#2c5282")
    ax.set_title("Prediction transitions: baseline LR → credits LR (test set)")
    ax.set_ylabel("Count")
    plt.xticks(rotation=15, ha="right")
    save_fig("12_error_transition_counts")

    ct = pd.crosstab(analysis_df["y_true"], analysis_df["transition"], normalize="index") * 100
    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.heatmap(ct, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax)
    ax.set_title("Transition mix (% within true class)")
    ax.set_ylabel("True class")
    save_fig("13_error_transition_by_true_label")

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(data=analysis_df, x="transition", y="talent_score", order=order, ax=ax, color="#bee3f8")
    ax.set_title("Talent score by transition bucket")
    plt.xticks(rotation=15, ha="right")
    save_fig("14_error_talent_score_by_transition")

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(data=analysis_df, x="transition", y="cast_size", order=order, ax=ax, color="#c6f6d5")
    ax.set_title("Cast size by transition bucket")
    plt.xticks(rotation=15, ha="right")
    save_fig("15_error_cast_size_by_transition")

    imp = analysis_df[analysis_df["transition"] == "wrong_to_correct"]
    reg = analysis_df[analysis_df["transition"] == "correct_to_wrong"]
    print("\nWrong→correct: mean franchise flag", round(imp["possible_franchise_flag"].mean(), 3), "| mean budget (M)", round(imp["budget"].mean() / 1e6, 2))
    print("Correct→wrong: mean franchise flag", round(reg["possible_franchise_flag"].mean(), 3), "| mean budget (M)", round(reg["budget"].mean() / 1e6, 2))


**Observation:** “Win” transitions often cluster where **talent / packaging** breaks ties ambiguous tabular features cannot resolve.  
**Business interpretation:** Credits are not a silver bullet — they **re-weight** evidence in borderline titles.  
**Modeling implication:** Monitor **correct→wrong** mass; if it grows with complexity, regularize or simplify talent composites.


## 4. Class-level error analysis

We report **precision / recall / F1** per class for **credits** and **baseline** on the same test rows, then inspect **confusion matrices** (counts + **row-normalized** “where true mass goes”).

**Reading guide:** “Average” vs “hit” confusion is common when **ROI buckets** are tight and pre-release features do not separate **moderate vs breakout** economics.

---

**Observation:** Macro metrics can hide **class-specific collapse**.  
**Business interpretation:** A studio cares differently about **missed flops** vs **missed hits** depending on the decision.  
**Modeling implication:** Per-class thresholds and **cost matrices** belong after this diagnostic.


In [ ]:
if analysis_df is None:
    print("Skip class analysis.")
else:
    yt = analysis_df["y_true"]
    pb = analysis_df["y_pred_baseline"]
    pc = analysis_df["y_pred_credits"]
    labels = sorted(set(yt) | set(pb) | set(pc))
    pref = ["flop", "average", "hit"]
    labels = [x for x in pref if x in labels] + [x for x in labels if x not in pref]

    print("\n===== BASELINE LR (TEST) =====")
    print(classification_report(yt, pb, digits=3, labels=labels, zero_division=0))
    print("\n===== CREDITS LR (TEST) =====")
    print(classification_report(yt, pc, digits=3, labels=labels, zero_division=0))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, pred, title in zip(axes, [pb, pc], ["Baseline LR", "Credits LR"]):
        cm = confusion_matrix(yt, pred, labels=labels)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
        ax.set_title(f"Confusion counts — {title}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    save_fig("16_error_confusion_counts_side_by_side")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, pred, title in zip(axes, [pb, pc], ["Baseline LR", "Credits LR"]):
        cm = confusion_matrix(yt, pred, labels=labels).astype(float)
        row = cm.sum(axis=1, keepdims=True)
        row[row == 0] = 1.0
        cmn = cm / row
        sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Purples", xticklabels=labels, yticklabels=labels, ax=ax, vmin=0, vmax=1)
        ax.set_title(f"Row-normalized confusion — {title}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    save_fig("17_error_confusion_normalized_side_by_side")

    f1b = f1_score(yt, pb, labels=labels, average=None, zero_division=0)
    f1c = f1_score(yt, pc, labels=labels, average=None, zero_division=0)
    sup = [int((yt == c).sum()) for c in labels]
    f1df = pd.DataFrame({"class": labels, "support": sup, "f1_baseline": f1b, "f1_credits": f1c})
    f1m = f1df.melt(id_vars=["class", "support"], value_vars=["f1_baseline", "f1_credits"], var_name="model", value_name="f1")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.barplot(data=f1m, x="class", y="f1", hue="model", ax=ax)
    ax.set_title("Per-class F1 (one-vs-rest on test)")
    ax.set_ylim(0, 1)
    save_fig("18_error_per_class_f1_comparison")
    display(f1df.round(3))


**Observation:** Normalized confusion rows show **where each true class leaks** in prediction space.  
**Business interpretation:** Mid-bucket (`average`) titles are often **economically ambiguous** with pre-release inputs.  
**Modeling implication:** Consider **ordinal** treatment or **hierarchical** models only if stakeholders accept added complexity.


## 5. Segment-level performance — genre, scale, budget, runtime

For each segment (e.g. `main_genre`), we compute **test accuracy** for baseline vs credits and the **delta**. We require **minimum support** (`n >= 25`) to avoid noisy micro-genres.

**Questions:** Which genres are “easy”? Where does credits help most? Does **large_scale** production make errors more or less likely?

---

**Observation:** Performance heterogeneity is the norm in entertainment tabular data.  
**Business interpretation:** Some genres are **data-rich** in this extract; others are **thin**.  
**Modeling implication:** Consider **segmented monitoring** in production, not a single global threshold.


In [ ]:
def segment_accuracy(g: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "n": len(g),
            "acc_baseline": accuracy_score(g["y_true"], g["y_pred_baseline"]),
            "acc_credits": accuracy_score(g["y_true"], g["y_pred_credits"]),
        }
    )


if analysis_df is None:
    print("Skip segment analysis.")
else:
    MIN_N = 25

    def run_segment(col: str, title: str, fname: str):
        agg = analysis_df.groupby(col).apply(segment_accuracy).reset_index()
        agg = agg[agg["n"] >= MIN_N].sort_values("acc_credits", ascending=False)
        agg["delta_acc"] = agg["acc_credits"] - agg["acc_baseline"]
        agg["n"] = agg["n"].astype(int)
        print(f"\n===== {title} (n>={MIN_N}) =====")
        display(agg.round(3).head(15))
        fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(agg.head(12)))))
        sub = agg.head(12).sort_values("acc_credits")
        ypos = np.arange(len(sub))
        ax.barh(ypos - 0.2, sub["acc_baseline"], height=0.35, label="baseline", color="#4a5568")
        ax.barh(ypos + 0.2, sub["acc_credits"], height=0.35, label="credits", color="#2b6cb0")
        ax.set_yticks(ypos)
        ax.set_yticklabels(sub[col].astype(str))
        ax.set_xlim(0, 1)
        ax.set_xlabel("Accuracy (test)")
        ax.set_title(title)
        ax.legend(loc="lower right")
        save_fig(fname)

    run_segment("main_genre", "Accuracy by main_genre", "19_error_segment_genre_accuracy")
    run_segment("production_scale", "Accuracy by production_scale", "20_error_segment_scale_accuracy")
    run_segment("budget_bucket", "Accuracy by budget_bucket (train tertiles on log1p budget)", "21_error_segment_budget_accuracy")
    run_segment("runtime_bucket", "Accuracy by runtime_bucket", "22_error_segment_runtime_accuracy")


**Observation:** Credits gains may **concentrate** in a handful of genres or budget bands.  
**Business interpretation:** That concentration tells you where the **data contract** is informative vs decorative.  
**Modeling implication:** If gains are isolated, prioritize **more features in those segments** (e.g. franchise metadata) rather than global complexity.


## 6. Confidence & probability behavior

We use **maximum predicted probability** as a simple **confidence proxy** (multiclass caveat: it is not a calibrated probability statement without additional calibration work).

We compare confidence for **correct vs incorrect** credits predictions, flag **low-confidence** (`max p < 0.4`) titles, and plot a **reliability-style** curve: for bins of predicted confidence, what fraction of rows are **actually correct**?

---

**Observation:** Miscalibrated confidence is a **product risk** (users trust scores too much).  
**Business interpretation:** Wide uncertain bands are an opportunity for **human review queues**.  
**Modeling implication:** If reliability curves are flat, prioritize **calibration** before new features.


In [ ]:
if analysis_df is None:
    print("Skip confidence analysis.")
else:
    ok = analysis_df["credits_correct"] == 1
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.histplot(analysis_df.loc[ok, "confidence"], color="#276749", label="correct", kde=True, stat="density", ax=ax)
    sns.histplot(analysis_df.loc[~ok, "confidence"], color="#c53030", label="incorrect", kde=True, stat="density", ax=ax)
    ax.set_title("Distribution of max predicted probability (credits LR)")
    ax.legend()
    save_fig("23_error_confidence_hist_correct_vs_wrong")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.boxplot(
        data=analysis_df.assign(outcome=np.where(analysis_df["credits_correct"], "correct", "incorrect")),
        x="outcome",
        y="confidence",
        ax=ax,
        hue="outcome",
        palette=["#276749", "#c53030"],
        legend=False,
    )
    ax.set_title("Confidence: correct vs incorrect (credits)")
    save_fig("24_error_confidence_boxplot")

    low_conf = analysis_df[analysis_df["confidence"] < 0.4]
    over_wrong = analysis_df[(analysis_df["confidence"] > 0.65) & (analysis_df["credits_correct"] == 0)]
    print("Low-confidence rows (<0.4):", len(low_conf), f"({100*len(low_conf)/len(analysis_df):.1f}%)")
    print("High-confidence wrong (>0.65):", len(over_wrong))

    bins = np.linspace(0, 1, 11)
    analysis_df = analysis_df.copy()
    analysis_df["conf_bin"] = pd.cut(analysis_df["confidence"], bins=bins, include_lowest=True)
    rel = analysis_df.groupby("conf_bin", observed=True).agg(mean_conf=("confidence", "mean"), acc=("credits_correct", "mean"), n=("credits_correct", "size")).reset_index()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(rel["mean_conf"], rel["acc"], marker="o", color="#2c5282")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.7)
    ax.set_xlabel("Mean predicted confidence (bin center ~)")
    ax.set_ylabel("Empirical accuracy (credits)")
    ax.set_title("Reliability-style curve (multiclass max-prob diagnostic)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    save_fig("25_error_reliability_curve_multiclass")

    display(rel.round(3))


**Observation:** Large gaps between the diagonal and the reliability line suggest **miscalibration** of the max-prob heuristic.  
**Business interpretation:** Low-confidence buckets are natural **“needs analyst”** queues in a Streamlit workflow.  
**Modeling implication:** `CalibratedClassifierCV` is the next professional step if probabilities enter decisions.


## 7. Case studies — cards for product / Streamlit

We auto-select a small set of **test** movies:

1. **Strong credits success** — credits correct & confidence ≥ 0.5  
2. **Strong credits failure** — credits wrong & confidence ≥ 0.55 (**risky** wrong)  
3. **Credits rescue** — `wrong_to_correct` with largest gain in **probability mass on the true class** vs baseline (baseline max prob for true class estimated via re-fit is heavy — we proxy rescue rank by **confidence** among wrong→correct)

For (3), we rank `wrong_to_correct` by **credits confidence** (simple, transparent).

Each card shows **title, truth, both predictions, key drivers** (genre, scale, talent_score, cast_size, franchise flag).

---

**Observation:** Narratives sell models; tables without “why” invite distrust.  
**Business interpretation:** Case cards translate logits into **story checks** a creative exec can challenge.  
**Modeling implication:** These rows become **golden fixtures** for UI regression testing later.


In [ ]:
def case_card(sub: pd.DataFrame, title: str) -> None:
    print("\n===", title, "===")
    cols = [
        "title",
        "y_true",
        "y_pred_baseline",
        "y_pred_credits",
        "confidence",
        "p_true_baseline",
        "p_true_credits",
        "delta_p_true",
        "main_genre",
        "production_scale",
        "budget_bucket",
        "runtime_bucket",
        "talent_score",
        "cast_size",
        "possible_franchise_flag",
    ]
    cols = [c for c in cols if c in sub.columns]
    display(sub[cols].round(3))


if analysis_df is None:
    print("Skip case studies.")
else:
    strong_ok = analysis_df[(analysis_df["credits_correct"] == 1) & (analysis_df["confidence"] >= 0.5)].sort_values(
        "confidence", ascending=False
    )
    strong_bad = analysis_df[(analysis_df["credits_correct"] == 0) & (analysis_df["confidence"] >= 0.55)].sort_values(
        "confidence", ascending=False
    )
    rescue = analysis_df[analysis_df["transition"] == "wrong_to_correct"].sort_values(
        ["delta_p_true", "confidence"], ascending=[False, False]
    )

    case_card(strong_ok.head(4), "Strong correct (credits, conf>=0.5)")
    case_card(strong_bad.head(4), "High-confidence failures (risk review)")
    case_card(rescue.head(5), "Credits rescue (wrong→correct, ranked by ΔP(true class) vs baseline)")


### How to read the case cards

- **Strong correct:** high confidence on the **true** class — candidates for “model got the packaging right” stories (still verify externally).  
- **High-confidence failures:** prioritize **risk review**; these are the most dangerous in UI if shown as “sure bets.”  
- **Credits rescues:** large **`delta_p_true`** means the credits feature block materially **reallocated** probability toward the realized bucket vs baseline — typical drivers include **talent_score**, **cast/crew scale**, and **director frequency** signals.

---

**Observation:** Case lists should be small and **defensible** — not exhaustive dumps.  
**Business interpretation:** These rows become **demo fixtures** for a Streamlit triage app.  
**Modeling implication:** Refresh cases whenever the feature contract or split policy changes.


## 8. Key findings & business interpretation

**What we learned (pattern-level, not over-claimed):**

- **Transitions** quantify whether credits features **repair** baseline errors or introduce **new** mistakes — both matter for governance.
- **Class-level** views show whether pain concentrates in **ambiguous middle outcomes** vs clear flops/hits.
- **Segments** (genre / scale / budget / runtime) expose **where** the model is allowed to be used with more vs less confidence.
- **Confidence** diagnostics highlight **UI risk**: overconfident wrong answers should trigger **disclaimers** or **human review**.

**Limitations (explicit):** TMDB credits omit marketing, competition, and true “star power” beyond name lists; **this PoC remains triage**, not a greenlight oracle.

---

**Observation:** Diagnostics usually reveal **structured failure** more than random noise.  
**Business interpretation:** Structured failure is **actionable** (features, segments, workflow).  
**Modeling implication:** Ship dashboards that show **confidence + segment** alongside the class label.
